# 04 — Evaluation: mAP, Dice Score & Dice Loss

**Goal**: Evaluate the trained YOLOv26-seg model on the held-out test set using:
1. **Ultralytics built-in metrics**: mAP@50, mAP@50-95, Precision, Recall
2. **Dice Score Coefficient (DSC)**: per-class and mean — measures segmentation overlap quality
3. **Dice Loss**: `1 - mean_DSC` — the corresponding loss value

### Metrics explained:
| Metric | Formula | What it measures |
|---|---|---|
| mAP@50 | mean AP at IoU=0.50 | Detection accuracy (bbox + mask) |
| mAP@50-95 | mean AP at IoU 0.50→0.95 | Stricter detection quality |
| Dice Score | 2|A∩B| / (|A|+|B|) | Pixel-wise segmentation overlap [0, 1] |
| Dice Loss | 1 - DSC | Optimization target for segmentation |

> **Prerequisite**: Run `03_training.ipynb` first — `runs/segment/yolo26s_run1/weights/best.pt` must exist.

## 1. Imports & Configuration

In [8]:
import numpy as np
import cv2
import json
import yaml
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.image as mpimg
import warnings
warnings.filterwarnings("ignore")

import torch
from ultralytics import YOLO

# Project root is one level up from notebooks/
BASE_DIR = Path("..")

# ── Paths ─────────────────────────────────────────────────────────────────
RUN_NAME     = "yolo26s_run1"   # must match 03_training.ipynb
DATASET_YAML = BASE_DIR / "dataset" / "dataset.yaml"
TEST_IMG_DIR = BASE_DIR / "dataset" / "images" / "test"
TEST_LBL_DIR = BASE_DIR / "dataset" / "labels" / "test"

# Auto-detect weights: Ultralytics resolves the project path relative to the
# notebook CWD, so outputs may land at notebooks/runs/runs/segment/... 
# rglob finds best.pt regardless of the exact nesting.
_hits = sorted(Path(".").rglob(f"{RUN_NAME}/weights/best.pt"))
assert _hits, (
    f"No best.pt found for run '{RUN_NAME}'.\n"
    f"Run 03_training.ipynb first, then re-run this cell."
)
WEIGHTS_PATH = _hits[0]

# Load dataset config
assert DATASET_YAML.exists(), (
    f"dataset.yaml not found at {DATASET_YAML}\n"
    f"Run 02_preprocessing.ipynb first."
)
with open(DATASET_YAML) as f:
    ds = yaml.safe_load(f)
CLASS_NAMES = ds["names"]
NUM_CLASSES  = ds["nc"]

print(f"Weights       : {WEIGHTS_PATH}")
print(f"Dataset yaml  : {DATASET_YAML.resolve()}")
print(f"Classes ({NUM_CLASSES})  : {CLASS_NAMES}")
print(f"Test images   : {len(list(TEST_IMG_DIR.glob('*.png')))}")
print(f"GPU available : {torch.cuda.is_available()}")


Weights       : runs\runs\segment\yolo26s_run1\weights\best.pt
Dataset yaml  : C:\NaufalFirdaus\CODES\MyGithub\self-driving-car-yolov26\dataset\dataset.yaml
Classes (11)  : ['class_1', 'class_2', 'class_3', 'class_4', 'class_5', 'class_6', 'class_7', 'class_8', 'class_9', 'class_10', 'class_11']
Test images   : 101
GPU available : True


## 2. Load Model

In [9]:
model  = YOLO(str(WEIGHTS_PATH))
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print(f"Model loaded from: {WEIGHTS_PATH}")
print(f"Running on      : {'GPU: ' + torch.cuda.get_device_name(0) if DEVICE == 0 else 'CPU'}")

Model loaded from: runs\runs\segment\yolo26s_run1\weights\best.pt
Running on      : GPU: NVIDIA GeForce RTX 4060 Laptop GPU


## 3. Built-in Metrics (mAP, Precision, Recall)
Ultralytics `model.val()` runs evaluation on the test split and returns detection and segmentation metrics.

In [10]:
metrics = model.val(
    data    = str(DATASET_YAML),
    split   = "test",
    imgsz   = 640,
    device  = DEVICE,
    conf    = 0.25,    # confidence threshold
    iou     = 0.50,    # IoU threshold for NMS
    verbose = True,
)

print("\n" + "=" * 50)
print("DETECTION METRICS (Bounding Box)")
print("-" * 50)
print(f"  mAP@50       : {metrics.box.map50:.4f}")
print(f"  mAP@50-95    : {metrics.box.map:.4f}")
print(f"  Precision    : {metrics.box.mp:.4f}")
print(f"  Recall       : {metrics.box.mr:.4f}")

print("\nSEGMENTATION METRICS (Mask)")
print("-" * 50)
print(f"  mAP@50       : {metrics.seg.map50:.4f}")
print(f"  mAP@50-95    : {metrics.seg.map:.4f}")
print(f"  Precision    : {metrics.seg.mp:.4f}")
print(f"  Recall       : {metrics.seg.mr:.4f}")
print("=" * 50)

Ultralytics 8.4.41  Python-3.12.10 torch-2.9.1+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO26s-seg summary (fused): 139 layers, 10,369,597 parameters, 0 gradients, 34.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 51.13.8 MB/s, size: 275.8 KB)
val: Scanning C:\NaufalFirdaus\CODES\MyGithub\self-driving-car-yolov26\dataset\labels\test... 101 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 101/101 423.0it/s 0.2s1s
val: New cache created: C:\NaufalFirdaus\CODES\MyGithub\self-driving-car-yolov26\dataset\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 2.6it/s 2.7s0.4s
                   all        101       2894      0.745      0.657      0.625      0.449      0.771      0.634      0.618      0.426
               class_1        101        619      0.726      0.437       0.42      0.191      0.774      0.391      0.404      0.254
      

## 4. Dice Score Coefficient & Dice Loss

### How it works:
1. Run YOLO `predict()` on each test image → get instance segmentation masks
2. **Merge** per-instance masks into a single semantic mask (pixel value = class ID)
3. Load the ground-truth semantic mask (original grayscale annotation)
4. Compute per-class binary Dice Score
5. Average across classes and images

**Dice Score = 2TP / (2TP + FP + FN)** — equivalent to F1 score on pixel level.

In [11]:
# ── Dice Score utilities ──────────────────────────────────────────────────

def instances_to_semantic_mask(result, img_h: int, img_w: int) -> np.ndarray:
    """
    Convert a YOLO segmentation Result (per-instance masks) into a
    semantic mask where each pixel holds a class ID (1-indexed; 0 = background).
    Later instances overwrite earlier ones at overlapping pixels.
    """
    semantic = np.zeros((img_h, img_w), dtype=np.uint8)
    if result.masks is None:
        return semantic

    masks   = result.masks.data.cpu().numpy()          # (N, h, w) float32
    classes = result.boxes.cls.cpu().numpy().astype(int)  # (N,) 0-indexed

    for mask, cls_id in zip(masks, classes):
        # Resize instance mask to original image size
        mask_resized = cv2.resize(mask, (img_w, img_h), interpolation=cv2.INTER_NEAREST)
        # class_id + 1 so background (0) stays intact
        semantic[mask_resized > 0.5] = cls_id + 1

    return semantic


def compute_dice_per_class(
    pred_semantic: np.ndarray,
    gt_semantic: np.ndarray,
    num_classes: int,
    class_ids_to_eval: list[int] | None = None,
) -> dict[int, float]:
    """
    Compute per-class Dice Score Coefficient.
    Both pred and gt use 1-indexed class IDs (0 = background, excluded).

    Returns: dict {class_id: dice_score}
    """
    if class_ids_to_eval is None:
        class_ids_to_eval = list(range(1, num_classes + 1))

    scores = {}
    for cls_id in class_ids_to_eval:
        pred_bin = (pred_semantic == cls_id).astype(float)
        gt_bin   = (gt_semantic   == cls_id).astype(float)

        intersection = (pred_bin * gt_bin).sum()
        denom        = pred_bin.sum() + gt_bin.sum()

        if denom == 0:
            scores[cls_id] = 1.0   # class absent in both pred & gt → perfect
        else:
            scores[cls_id] = (2.0 * intersection) / denom

    return scores


def dice_loss(dice_scores: dict[int, float]) -> float:
    """Dice Loss = 1 - mean Dice Score across all classes."""
    return 1.0 - float(np.mean(list(dice_scores.values())))


print("Dice Score utilities defined.")

Dice Score utilities defined.


## 5. Load Ground-Truth Masks
We read the original grayscale masks (pixel value = class ID) to compare with YOLO predictions.

In [12]:
# Original test masks (pixel value = class ID, 1-indexed classes; 0 = background)
ORIG_TEST_MASK_DIR = next(BASE_DIR.glob("annotations_prepped_test*/*"))

test_images = sorted(TEST_IMG_DIR.glob("*.png"))
print(f"Test images: {len(test_images)}")
print(f"GT masks   : {ORIG_TEST_MASK_DIR}")

# Verify GT masks exist for all test images
missing_gt = [p for p in test_images if not (ORIG_TEST_MASK_DIR / p.name).exists()]
if missing_gt:
    print(f"⚠️  Missing GT masks for {len(missing_gt)} images")
else:
    print("✓ All test images have ground truth masks")

Test images: 101
GT masks   : ..\annotations_prepped_test-20230811T065240Z-001\annotations_prepped_test
✓ All test images have ground truth masks


## 6. Run Inference & Compute Dice Scores
This processes all test images and accumulates per-class Dice scores.

In [13]:
IMG_H, IMG_W  = 360, 480   # native image resolution
CLASS_IDS     = list(range(1, NUM_CLASSES + 1))  # 1-indexed (0 = background)
CONF_THRESHOLD = 0.25

# Accumulate per-image Dice scores
all_dice_scores = {cls_id: [] for cls_id in CLASS_IDS}
n_processed = 0

for img_path in test_images:
    gt_mask_path = ORIG_TEST_MASK_DIR / img_path.name
    if not gt_mask_path.exists():
        continue

    # Ground-truth semantic mask
    gt_mask = np.array(Image.open(gt_mask_path).convert("L"))

    # YOLO prediction
    result = model.predict(
        str(img_path),
        imgsz=640,
        conf=CONF_THRESHOLD,
        device=DEVICE,
        verbose=False,
    )[0]

    # Convert instance predictions → semantic mask
    pred_mask = instances_to_semantic_mask(result, IMG_H, IMG_W)

    # Per-class Dice
    dice_scores = compute_dice_per_class(pred_mask, gt_mask, NUM_CLASSES, CLASS_IDS)
    for cls_id, score in dice_scores.items():
        all_dice_scores[cls_id].append(score)

    n_processed += 1
    if n_processed % 20 == 0 or n_processed == len(test_images):
        print(f"  Processed: {n_processed}/{len(test_images)}", end="\r")

print(f"\n✓ Inference complete. Processed {n_processed} images.")

  Processed: 101/101
✓ Inference complete. Processed 101 images.


## 7. Dice Score Results

In [14]:
# Per-class mean Dice
mean_dice_per_class = {
    cls_id: float(np.mean(scores))
    for cls_id, scores in all_dice_scores.items()
    if scores
}

overall_dice  = float(np.mean(list(mean_dice_per_class.values())))
overall_dloss = 1.0 - overall_dice

print("=" * 55)
print(f"{'Class ID':<12} {'Class Name':<22} {'Dice Score':>12}")
print("-" * 55)
for cls_id, score in sorted(mean_dice_per_class.items()):
    name = CLASS_NAMES[cls_id - 1] if (cls_id - 1) < len(CLASS_NAMES) else f"class_{cls_id}"
    bar  = "█" * int(score * 20)
    print(f"{cls_id:<12} {name:<22} {score:>10.4f}  {bar}")
print("-" * 55)
print(f"{'Mean Dice Score':<34} {overall_dice:>10.4f}")
print(f"{'Dice Loss (1 - DSC)':<34} {overall_dloss:>10.4f}")
print("=" * 55)

Class ID     Class Name               Dice Score
-------------------------------------------------------
1            class_1                    0.9009  ██████████████████
2            class_2                    0.1660  ███
3            class_3                    0.9152  ██████████████████
4            class_4                    0.9373  ██████████████████
5            class_5                    0.9474  ██████████████████
6            class_6                    0.7694  ███████████████
7            class_7                    0.7434  ██████████████
8            class_8                    0.8507  █████████████████
9            class_9                    0.6451  ████████████
10           class_10                   0.9222  ██████████████████
11           class_11                   0.3615  ███████
-------------------------------------------------------
Mean Dice Score                        0.7417
Dice Loss (1 - DSC)                    0.2583


In [15]:
# Per-class Dice Score bar chart
names  = [CLASS_NAMES[cid-1] if (cid-1) < len(CLASS_NAMES) else f"class_{cid}"
          for cid in sorted(mean_dice_per_class)]
scores = [mean_dice_per_class[cid] for cid in sorted(mean_dice_per_class)]
colors = ["#2ecc71" if s >= 0.7 else "#f39c12" if s >= 0.4 else "#e74c3c" for s in scores]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(names, scores, color=colors, edgecolor="#333")
ax.axvline(overall_dice, color="navy", linestyle="--", linewidth=1.5, label=f"Mean DSC = {overall_dice:.3f}")
ax.set_xlim(0, 1.05)
ax.set_xlabel("Dice Score Coefficient", fontsize=12)
ax.set_title("Per-Class Dice Score on Test Set", fontsize=14)
ax.legend()

for bar, score in zip(bars, scores):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f"{score:.3f}", va="center", fontsize=10)

# Color legend
legend_info = [
    mpatches.Patch(color="#2ecc71", label="Good (≥ 0.70)"),
    mpatches.Patch(color="#f39c12", label="Fair (0.40–0.70)"),
    mpatches.Patch(color="#e74c3c", label="Poor (< 0.40)"),
]
ax.legend(handles=legend_info + [plt.Line2D([0], [0], color="navy", linestyle="--",
          label=f"Mean DSC = {overall_dice:.3f}")])

plt.tight_layout()
plt.savefig("dice_scores.png", dpi=150)
plt.show()
print("Saved: dice_scores.png")

<Figure size 1000x500 with 1 Axes>

Saved: dice_scores.png


## 8. Full Metrics Summary

In [16]:
summary = {
    "detection": {
        "mAP50"    : round(float(metrics.box.map50), 4),
        "mAP50_95" : round(float(metrics.box.map),   4),
        "precision": round(float(metrics.box.mp),    4),
        "recall"   : round(float(metrics.box.mr),    4),
    },
    "segmentation": {
        "mAP50"      : round(float(metrics.seg.map50), 4),
        "mAP50_95"   : round(float(metrics.seg.map),   4),
        "precision"  : round(float(metrics.seg.mp),    4),
        "recall"     : round(float(metrics.seg.mr),    4),
        "mean_dice"  : round(overall_dice,  4),
        "dice_loss"  : round(overall_dloss, 4),
        "per_class_dice": {
            CLASS_NAMES[cid-1] if (cid-1) < len(CLASS_NAMES) else f"class_{cid}": round(v, 4)
            for cid, v in mean_dice_per_class.items()
        },
    },
    "model": str(WEIGHTS_PATH),
    "test_images": n_processed,
}

out_path = BASE_DIR / "evaluation_results.json"
with open(out_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"Results saved to: {out_path}")
print("\n=" * 50)
print("FINAL RESULTS SUMMARY")
print("=" * 50)
print(f"  Seg mAP@50       : {summary['segmentation']['mAP50']}")
print(f"  Seg mAP@50-95    : {summary['segmentation']['mAP50_95']}")
print(f"  Mean Dice Score  : {summary['segmentation']['mean_dice']}")
print(f"  Dice Loss        : {summary['segmentation']['dice_loss']}")
print("=" * 50)

Results saved to: ..\evaluation_results.json

=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
=
FINAL RESULTS SUMMARY
  Seg mAP@50       : 0.6183
  Seg mAP@50-95    : 0.4265
  Mean Dice Score  : 0.7417
  Dice Loss        : 0.2583


## 9. Visual Prediction Comparison
Side-by-side: original image | ground truth mask | predicted mask

In [17]:
PALETTE = [
    (0,0,0), (128,64,128), (70,70,70), (0,0,142), (220,20,60),
    (107,142,35), (0,130,180), (153,153,190), (255,128,0), (0,60,100),
]

def colorize(mask: np.ndarray) -> np.ndarray:
    rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for v in np.unique(mask):
        c = PALETTE[v % len(PALETTE)]
        rgb[mask == v] = c
    return rgb

# Pick 4 random test samples
import random
random.seed(42)
sample_imgs = random.sample(list(test_images), min(4, len(test_images)))

fig, axes = plt.subplots(4, 3, figsize=(15, 17))

for i, img_path in enumerate(sample_imgs):
    gt_mask_path = ORIG_TEST_MASK_DIR / img_path.name
    img     = np.array(Image.open(img_path))
    gt_mask = np.array(Image.open(gt_mask_path).convert("L")) if gt_mask_path.exists() else np.zeros((IMG_H, IMG_W))

    result   = model.predict(str(img_path), imgsz=640, conf=CONF_THRESHOLD, device=DEVICE, verbose=False)[0]
    pred_sem = instances_to_semantic_mask(result, IMG_H, IMG_W)

    # Dice for this image
    d = compute_dice_per_class(pred_sem, gt_mask, NUM_CLASSES, CLASS_IDS)
    img_dice = np.mean(list(d.values()))

    axes[i][0].imshow(img)
    axes[i][0].set_title(f"Image", fontsize=9)
    axes[i][1].imshow(colorize(gt_mask))
    axes[i][1].set_title("Ground Truth", fontsize=9)
    axes[i][2].imshow(colorize(pred_sem))
    axes[i][2].set_title(f"Prediction  (Dice={img_dice:.3f})", fontsize=9)
    for ax in axes[i]:
        ax.axis("off")

legend_patches = [mpatches.Patch(color=[c/255 for c in PALETTE[v % len(PALETTE)]],
                  label=f"Class {v}") for v in range(NUM_CLASSES + 1)]
fig.legend(handles=legend_patches, loc="lower center", ncol=NUM_CLASSES + 1, fontsize=8)

plt.suptitle("Prediction vs Ground Truth (Test Set Samples)", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("prediction_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: prediction_comparison.png")
print("\n✓ Evaluation complete. Proceed to 05_hyperparameter_tuning.ipynb if metrics need improvement.")

<Figure size 1500x1700 with 12 Axes>

Saved: prediction_comparison.png

✓ Evaluation complete. Proceed to 05_hyperparameter_tuning.ipynb if metrics need improvement.
